# Landau Damping

Landau damping is the collisionless damping of electrostatic plasma waves arising from wave–particle resonance. The **Vlasov–Poisson system** is evolved for a 1D1v distribution $f(x,v,t)$:

$$\partial_t f + v\,\partial_x f + E\,\partial_v f = 0, \qquad \partial_x E = \rho - 1$$

A cosine perturbation $n(x,0)=1+\epsilon\cos(kx)$ excites a Langmuir wave. The electric field energy $\langle E_x^2\rangle$ decays exponentially at the Landau rate $\gamma_L$, which can be compared against the analytic prediction from the plasma dispersion function $Z(\zeta)$.

In [ ]:
include("../scripts/select_backend.jl")
bslLD.greet()

In [ ]:
using Statistics, CairoMakie, Base64

## Simulation Setup

The phase space is discretised on a $128\times128$ grid with $x\in[0,4\pi]$ and $v\in[-7,7]$, using a thermal velocity $v_{th}=1$. The density is perturbed by $\epsilon=10^{-4}$ with wave number $k=2\pi/(4\pi)=0.5$. A drifting beam component is added at $v_d=4\,v_{th}$ to seed nonlinear trapping.

In [ ]:
grid =  bslLD.Grid([0.0,-7.0],[4*pi,7.0],[128,128],1, 0.0, 1)
simTime = bslLD.SimulationTime(0.01, 100.0; nmax=10000)

# initFuncv(v)= exp(-(v+2)^2 / 2) / sqrt(2*pi)+ exp(-(v-2)^2 / 2) / sqrt(2*pi)
initFuncv(v) = exp(-v^2 / 2) / sqrt(2*pi)
initFuncv(v) = exp(-v^2 / 2) / sqrt(2*pi) + 0.4*exp(-(v-4)^2 / 2) / sqrt(2*pi)

f = bslLD.Distribution(grid, 0.0001,initFuncv=initFuncv);
e = bslLD.empty_vectorfield(grid);


In [ ]:
mutable struct Diag
    rho::Vector
    f::Vector
    Ex::Vector
end
Diag() = Diag([], [], [])

function diags!(diags, f, rho, Ex, grid, simTime)
    simTime.step % 10 == 0 || return
    push!(diags.f, copy(f.data .- mean(f.data, dims=1)))
    push!(diags.rho, copy(rho.data[:]))
    push!(diags.Ex, copy(Ex.data[:]))
end

function step!(f, grid, simTime, diags)
    bslLD.advectX!(f, grid, simTime)
    rho = bslLD.compute_density(f, grid)
    sol = bslLD.solve_fields(bslLD.Moments(rho), grid, bslLD.PoissonSolver())
    bslLD.advectV!(f, grid, simTime, sol.E)
    diags!(diags, f, rho, sol.E[1], grid, simTime)
end;


## Time Evolution

A first-order Lie–Trotter splitting advances the Vlasov equation: an X-advection step shifts $f$ along characteristics in $x$, a Poisson solve updates $E_x[\rho]$, then a V-advection step shifts $f$ along $v$. Each time step is $\Delta t=0.01$; the simulation runs to $T=100$.

In [ ]:
diags = Diag()
while bslLD.continue_advection(simTime, true)
    step!(f, grid, simTime, diags)
    bslLD.advance!(simTime)
end


## Distribution Function Evolution

The heatmap shows $f(x,v,t)$ at successive times. Phase-space vortices (holes) develop as the wave traps resonant particles near $v\approx\omega/k$.

In [ ]:
num_frames = length(diags.f)

data_obs  = Observable(transpose(Array(diags.f[1])))
title_obs = Observable("Frame 1")

fig = Figure(size = (600, 500))
ax  = Axis(fig[1, 1], xlabel = "x", ylabel = "v", title = title_obs)
hm  = heatmap!(ax, data_obs)
Colorbar(fig[1, 2], hm)

record(fig, "fdiag_heatmap_animation.gif", 1:num_frames; framerate = 10) do i
    data_obs[]  = transpose(Array(diags.f[i]))
    title_obs[] = "Frame $i"
end

gif_bytes = read("fdiag_heatmap_animation.gif")
b64 = Base64.base64encode(gif_bytes)
HTML("<img src=\"data:image/gif;base64,$(b64)\" />")

## Electric Field Energy

The spatial variance of $\rho(x,t)$ tracks the electrostatic wave energy $\langle E_x^2\rangle$. The semi-log plot reveals the exponential Landau damping rate $\gamma_L$.

In [ ]:
fig = Figure()
ax  = Axis(fig[1, 1], xlabel = "step", ylabel = "⟨ρ²⟩", yscale = log10)
lines!(ax, map(x -> mean((x .- mean(x)).^2), Array.(diags.rho)))
fig

In [ ]:
using FFTW

## Frequency Spectrum

FFT of the final $E_x(k)$ field confirms the dominant excited mode at wavenumber $k=2\pi/L_x$.

In [ ]:
fig = Figure()
ax  = Axis(fig[1, 1], xlabel = "mode", ylabel = "log|FFT(Eₓ)|")
lines!(ax, log.(abs.(fft(diags.Ex[end]))))
fig